Модель регрессии, основанная на билиотеке Catboost

In [38]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

df = pd.read_csv("Housing.csv")
df.head()


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [40]:
df.columns = df.columns.str.strip()
df['total_rooms'] = df['bedrooms'] + df['bathrooms']
df['area_per_room'] = df['area'] / (df['bedrooms'] + 1)  # +1 чтобы не делить на 0
X = df.drop('price', axis=1)
y = df['price']


In [42]:
# делим данные
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [43]:
# задание параметров для модели
model = CatBoostRegressor(
    iterations=1500,        # ↑ больше деревьев
    learning_rate=0.05,     # ↓ меньше шаг → стабильнее
    depth=6,                # ↑ глубже деревья → ловит сложные зависимости
    # l2_leaf_reg=5.0,        # ↑ регуляризация против переобучения
    loss_function='RMSE',
    eval_metric='R2',
    early_stopping_rounds=100,
    verbose=100,
    random_seed=42,
    cat_features=['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']
)

In [44]:
X_train.dtypes

area                  int64
bedrooms              int64
bathrooms             int64
stories               int64
mainroad             object
guestroom            object
basement             object
hotwaterheating      object
airconditioning      object
parking               int64
prefarea             object
furnishingstatus     object
total_rooms           int64
area_per_room       float64
dtype: object

In [45]:
X_train.head()

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus,total_rooms,area_per_room
46,6000,3,2,4,yes,no,no,no,yes,1,no,furnished,5,1500.0
93,7200,3,2,1,yes,no,yes,no,yes,3,no,semi-furnished,5,1800.0
335,3816,2,1,1,yes,no,yes,no,yes,2,no,furnished,3,1272.0
412,2610,3,1,2,yes,no,yes,no,no,0,yes,unfurnished,4,652.5
471,3750,3,1,2,yes,no,no,no,no,0,no,unfurnished,4,937.5


In [46]:
model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	learn: 0.0485289	test: 0.0160678	best: 0.0160678 (0)	total: 23.5ms	remaining: 35.3s
100:	learn: 0.7885808	test: 0.6032827	best: 0.6032827 (100)	total: 2.27s	remaining: 31.4s
200:	learn: 0.8676137	test: 0.6331063	best: 0.6332701 (199)	total: 4.43s	remaining: 28.6s
300:	learn: 0.9050200	test: 0.6372118	best: 0.6372118 (300)	total: 6.51s	remaining: 25.9s
400:	learn: 0.9268475	test: 0.6400213	best: 0.6407665 (376)	total: 8.6s	remaining: 23.6s
500:	learn: 0.9393446	test: 0.6415416	best: 0.6424758 (459)	total: 10.6s	remaining: 21.2s
600:	learn: 0.9489567	test: 0.6422734	best: 0.6437173 (583)	total: 12.6s	remaining: 18.9s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6437173097
bestIteration = 583

Shrink model to first 584 iterations.


CatBoostRegressor(cat_features=['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus'], depth=6, early_stopping_rounds=100, eval_metric='R2', iterations=1500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [47]:
# Посмотреть важность признаков
fi = pd.DataFrame({
    'feature': X.columns.tolist(),
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False)

print("🏆 Топ-10 признаков:")
print(fi.head(10))

🏆 Топ-10 признаков:
             feature  importance
0               area   24.066022
12       total_rooms   10.641758
2          bathrooms    9.648601
11  furnishingstatus    9.217122
8    airconditioning    7.565953
9            parking    7.233852
3            stories    7.118940
13     area_per_room    6.850984
6           basement    5.786459
10          prefarea    4.518621


In [48]:
predict = model.predict(X_test)
print(f"r2:{r2_score(y_test, predict):.3f}")
print(f"RMSE:{np.sqrt(mean_absolute_error(y_test, predict)):.3f}")
print(f"MAE:{mean_absolute_error(y_test, predict):.3f}")

r2:0.644
RMSE:993.522
MAE:987085.856


In [49]:
model.save_model('catboost_house_price.cbm')

In [ ]:
# Загрузка потом: model = CatBoostRegressor(); model.load_model('catboost_model.cbm')